In [1]:
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
from IPython.display import HTML, display
from train import train_agent
import yaml
from env import AsteroidDefenseEnv

def render(env, agent, steps=200):
    s, _ = env.reset()

    for _ in range(steps):
        a = agent.act(s)
        s, _, d, _, _ = env.step(a)

        plt.figure(figsize=(6, 6))
        plt.xlim(0, 1)
        plt.ylim(0, 1)

        # asteroids
        for ast in env.asteroids:
            sx, sy = env.project_to_screen01(ast["pos"])
            depth = np.clip(ast["pos"][2] / env.max_distance, 0.0, 1.0)
            size = 30 + 120 * (1.0 - depth)
            plt.scatter([sx], [sy], c='black', s=size)

        # projectiles
        for p in env.projectiles:
            sx, sy = env.project_to_screen01(p["pos"])
            plt.scatter([sx], [sy], c='orange', s=12)

        # gun direction
        gun_tip = env._gun_direction() * 2.0
        gx, gy = env.project_to_screen01(gun_tip)
        plt.plot([0.5, gx], [0.5, gy], c='blue')

        plt.gca().set_aspect('equal')
        plt.pause(0.01)
        plt.close()

        if d:
            break

def render_animation(env, agent, steps=200, interval=50, save_path=None, show=True):
    s, _ = env.reset()
    frames = []

    for _ in range(steps):
        a = agent.act(s)
        s, _, d, _, _ = env.step(a)

        ast_offsets = []
        ast_sizes = []
        for ast in env.asteroids:
            sx, sy = env.project_to_screen01(ast["pos"])
            depth = np.clip(ast["pos"][2] / env.max_distance, 0.0, 1.0)
            size = 30 + 120 * (1.0 - depth)
            ast_offsets.append([sx, sy])
            ast_sizes.append(size)

        proj_offsets = []
        for p in env.projectiles:
            sx, sy = env.project_to_screen01(p["pos"])
            proj_offsets.append([sx, sy])

        gun_tip = env._gun_direction() * 2.0
        gx, gy = env.project_to_screen01(gun_tip)

        frames.append((
            np.array(ast_offsets, dtype=np.float32),
            np.array(ast_sizes, dtype=np.float32),
            np.array(proj_offsets, dtype=np.float32),
            (gx, gy),
        ))

        if d:
            break

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')

    ast_scatter = ax.scatter([], [], c='black', s=[])
    proj_scatter = ax.scatter([], [], c='orange', s=12)
    gun_line, = ax.plot([], [], c='blue')

    def init():
        ast_scatter.set_offsets(np.empty((0, 2)))
        ast_scatter.set_sizes(np.array([]))
        proj_scatter.set_offsets(np.empty((0, 2)))
        gun_line.set_data([], [])
        return ast_scatter, proj_scatter, gun_line

    def update(i):
        ast_offsets, ast_sizes, proj_offsets, gun_tip = frames[i]
        ast_scatter.set_offsets(ast_offsets)
        ast_scatter.set_sizes(ast_sizes)
        proj_scatter.set_offsets(proj_offsets)
        gun_line.set_data([0.5, gun_tip[0]], [0.5, gun_tip[1]])
        return ast_scatter, proj_scatter, gun_line

    anim = animation.FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=len(frames),
        interval=interval,
        blit=True,
    )

    if save_path:
        fps = 1000 / interval if interval > 0 else 20
        anim.save(save_path, fps=fps)

    if show:
        try:
            display(HTML(anim.to_jshtml()))
        except Exception:
            pass

    return anim


In [3]:
agent = train_agent()
env = CameraEnv(yaml.safe_load(open("config.yaml"))["env"])
render_animation(env, agent)

ep 0 reward -109.50
ep 1 reward -78.39
ep 2 reward -63.22


KeyboardInterrupt: 